In [41]:
import os
os.environ["ANTHROPIC_API_KEY"] = "paste-your-key-here"
#!pip install anthropic
"""
I'm using manual inputs to demonstrate the logic but in prod we'd connect to live market data so Greeks reprice in real time.
"""

"\nI'm using manual inputs to demonstrate the logic but in prod we'd connect to live market data so Greeks reprice in real time.\n"

In [47]:
from scipy.stats import norm
import numpy as np

def BS_greeks(S, K, T, r, sigma, option_type='call'):
    """
    S = current spot price
    K = strike price
    T = time to expiry in years
    r = risk free rate
    sigma = implied volatility (e.g. 0.3 = 30%)
    option_type = 'call' or 'put'
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1

    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * norm.pdf(d1) * np.sqrt(T) * 0.01  # per 1% vol move
    theta = (-S * norm.pdf(d1) * sigma / (2 * np.sqrt(T))) / 365  # per day

    return {
        "delta": round(delta, 4),
        "gamma": round(gamma, 4),
        "vega":  round(vega, 4),
        "theta": round(theta, 4)
    }

print("Black-Scholes Greeks function loaded")

Black-Scholes Greeks function loaded


In [48]:
from datetime import datetime
#input a new trade

spot_price    = 108.0   # Brent crude current price
strike_price  = 120.0   # OTM betting on further spike
implied_vol   = 0.55    # vol has exploded with iran sanctioning back 
risk_free_rate = 0.05
expiry_date   = "2026-06-30"
option_type   = "call"
position_size = 100
direction     = "long"    

today = datetime.today()
expiry = datetime.strptime(expiry_date, "%Y-%m-%d")
T = (expiry - today).days / 252 

print(f"Position: {direction} {position_size} {option_type} options")
print(f"Spot: ${spot_price} | Strike: ${strike_price} | Expiry: {expiry_date}")
print(f"Implied Vol: {implied_vol*100}% | Risk Free Rate: {risk_free_rate*100}%")
print(f"Time to expiry: {round(T, 4)} years ({(expiry - today).days} trading days)")

Position: long 100 call options
Spot: $108.0 | Strike: $120.0 | Expiry: 2026-06-30
Implied Vol: 55.00000000000001% | Risk Free Rate: 5.0%
Time to expiry: 0.3889 years (98 trading days)


In [49]:
# Calc Greeks
greeks = BS_greeks(
    S           = spot_price,
    K           = strike_price,
    T           = T,
    r           = risk_free_rate,
    sigma       = implied_vol,
    option_type = option_type
)

# Adjust for direction
multiplier = 1 if direction == "long" else -1

print("=== GREEKS ===")
print(f"Delta : {round(greeks['delta'] * multiplier, 3)}")
print(f"Gamma : {round(greeks['gamma'] * multiplier, 3)}")
print(f"Vega  : {round(greeks['vega']  * multiplier, 3)}")
print(f"Theta : {round(greeks['theta'] * multiplier, 3)}")

=== GREEKS ===
Delta : 0.468
Gamma : 0.011
Vega  : 0.268
Theta : -0.052


In [50]:
import anthropic
import json

def generate_risk_report(position_details, greeks):
    
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    
    context = f"""
    I'm a risk analyst at Maven
    
    A trader wants to enter this position:
    - Asset: Crude Oil Options
    - Direction: {position_details['direction']}
    - Option type: {position_details['option_type']}
    - Position size: {position_details['size']} contracts
    - Spot price: ${position_details['spot']}
    - Strike price: ${position_details['strike']}
    - Expiry: {position_details['expiry']}
    - Implied volatility: {position_details['vol']*100}%
    
    The computed Greeks are:
    - Delta: {greeks['delta']}
    - Gamma: {greeks['gamma']}
    - Vega: {greeks['vega']}
    - Theta: {greeks['theta']}
    
    Write a concise plain English risk summary for the risk manager.
    Cover: directional risk, vol risk, time decay, and any key concerns.
    Do not recompute or second guess the numbers above.
    Keep it to 4-5 bullet points
    """
    
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role": "user", "content": context}]
    )
    
    return message.content[0].text

print("Risk report function loaded")

Risk report function loaded


In [51]:
# Package position details into a dictionary
position_details = {
    'direction':   direction,
    'option_type': option_type,
    'size':        position_size,
    'spot':        spot_price,
    'strike':      strike_price,
    'expiry':      expiry_date,
    'vol':         implied_vol
}

# Generate the risk report
print("=== LLM RISK REPORT ===")
print("Generating report...\n")

report = generate_risk_report(position_details, greeks)
print(report)

print("\n=== RAW GREEKS ===")
print(f"Delta : {greeks['delta']}")
print(f"Gamma : {greeks['gamma']}")
print(f"Vega  : {greeks['vega']}")
print(f"Theta : {greeks['theta']}")

=== LLM RISK REPORT ===
Generating report...

**Risk Summary - Crude Oil Call Options Position**

• **Directional Risk**: Position has moderate oil price sensitivity with 46.85 delta exposure per contract (4,685 total). For every $1 move in crude oil, the position gains/loses ~$4,685. The high gamma (1.07) means this exposure will increase rapidly if oil rallies toward the $120 strike.

• **Volatility Risk**: Significant exposure with 26.78 vega per contract (2,678 total). A 1% drop in implied volatility from current 55% levels would cost ~$2,678. Given the high current vol, the position is vulnerable to volatility compression.

• **Time Decay**: Daily theta bleed of $51.90 per contract ($5,190 total) will accelerate as we approach expiry in mid-2026. This represents meaningful carrying cost that must be overcome by favorable oil price or volatility moves.

• **Key Concerns**: Options are currently out-of-the-money by $12 (~11% above spot), requiring substantial oil rally to reach prof